## LangGraph Open Deep Research - Supervisor-Researcher Architecture

In this notebook, we'll explore the **supervisor-researcher delegation architecture** for conducting deep research with LangGraph.

You can visit this repository to see the original application: [Open Deep Research](https://github.com/langchain-ai/open_deep_research)

Let's jump in!

## What We're Building

This implementation uses a **hierarchical delegation pattern** where:

1. **User Clarification** - Optionally asks clarifying questions to understand the research scope
2. **Research Brief Generation** - Transforms user messages into a structured research brief
3. **Supervisor** - A lead researcher that analyzes the brief and delegates research tasks
4. **Parallel Researchers** - Multiple sub-agents that conduct focused research simultaneously
5. **Research Compression** - Each researcher synthesizes their findings
6. **Final Report** - All findings are combined into a comprehensive report

![Architecture Diagram](https://private-user-images.githubusercontent.com/181020547/465824799-12a2371b-8be2-4219-9b48-90503eb43c69.png?jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmF3LmdpdGh1YnVzZXJjb250ZW50LmNvbSIsImtleSI6ImtleTUiLCJleHAiOjE3NjAwNDgyMzcsIm5iZiI6MTc2MDA0NzkzNywicGF0aCI6Ii8xODEwMjA1NDcvNDY1ODI0Nzk5LTEyYTIzNzFiLThiZTItNDIxOS05YjQ4LTkwNTAzZWI0M2M2OS5wbmc_WC1BbXotQWxnb3JpdGhtPUFXUzQtSE1BQy1TSEEyNTYmWC1BbXotQ3JlZGVudGlhbD1BS0lBVkNPRFlMU0E1M1BRSzRaQSUyRjIwMjUxMDA5JTJGdXMtZWFzdC0xJTJGczMlMkZhd3M0X3JlcXVlc3QmWC1BbXotRGF0ZT0yMDI1MTAwOVQyMjEyMTdaJlgtQW16LUV4cGlyZXM9MzAwJlgtQW16LVNpZ25hdHVyZT1iYTRmYTAzYjkzYjA2MGE4ZTZlYjQ4ODU1OWIwY2VlZWU0Mzk0YzdmMjQ1YTlhMDMyNmI3NWNlZTQxNDdlZGViJlgtQW16LVNpZ25lZEhlYWRlcnM9aG9zdCJ9.a8477QD1J4Lrmys7jB8gt_H5pdiKBsKsu3npEqZjEpo)

This differs from a section-based approach by allowing dynamic task decomposition based on the research question, rather than predefined sections.

## Dependencies

You'll need API keys for Anthropic (for the LLM) and Tavily (for web search). We'll configure the system to use Anthropic's Claude Sonnet 4 exclusively.

In [1]:
import os
import getpass

os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")
os.environ["TAVILY_API_KEY"] = getpass.getpass("Enter your Tavily API key: ")

## Task 1: State Definitions

The state structure is hierarchical with three levels:

### Agent State (Top Level)
Contains the overall conversation messages, research brief, accumulated notes, and final report.

### Supervisor State (Middle Level)
Manages the research supervisor's messages, research iterations, and coordinating parallel researchers.

### Researcher State (Bottom Level)
Each individual researcher has their own message history, tool call iterations, and research findings.

We also have structured outputs for tool calling:
- **ConductResearch** - Tool for supervisor to delegate research to a sub-agent
- **ResearchComplete** - Tool to signal research phase is done
- **ClarifyWithUser** - Structured output for asking clarifying questions
- **ResearchQuestion** - Structured output for the research brief

Let's import these from our library: [`open_deep_library/state.py`](open_deep_library/state.py)

In [2]:
# Import state definitions from the library
from open_deep_library.state import (
    # Main workflow states
    AgentState,           # Lines 65-72: Top-level agent state with messages, research_brief, notes, final_report
    AgentInputState,      # Lines 62-63: Input state is just messages
    
    # Supervisor states
    SupervisorState,      # Lines 74-81: Supervisor manages research delegation and iterations
    
    # Researcher states
    ResearcherState,      # Lines 83-90: Individual researcher with messages and tool iterations
    ResearcherOutputState, # Lines 92-96: Output from researcher (compressed research + raw notes)
    
    # Structured outputs for tool calling
    ConductResearch,      # Lines 15-19: Tool for delegating research to sub-agents
    ResearchComplete,     # Lines 21-22: Tool to signal research completion
    ClarifyWithUser,      # Lines 30-41: Structured output for user clarification
    ResearchQuestion,     # Lines 43-48: Structured output for research brief
)

#### ❓ Question 1:

 Explain the interrelationships between the three states.  Why don't we just make a single huge state?

## Task 2: Utility Functions and Tools

The system uses several key utilities:

### Search Tools
- **tavily_search** - Async web search with automatic summarization to stay within token limits
- Supports Anthropic native web search and Tavily API

### Reflection Tools
- **think_tool** - Allows researchers to reflect on their progress and plan next steps (ReAct pattern)

### Helper Utilities
- **get_all_tools** - Assembles the complete toolkit (search + MCP + reflection)
- **get_today_str** - Provides current date context for research
- Token limit handling utilities for graceful degradation

These are defined in [`open_deep_library/utils.py`](open_deep_library/utils.py)

In [3]:
# Import utility functions and tools from the library
from open_deep_library.utils import (
    # Search tool - Lines 43-136: Tavily search with automatic summarization
    tavily_search,
    
    # Reflection tool - Lines 219-244: Strategic thinking tool for ReAct pattern
    think_tool,
    
    # Tool assembly - Lines 569-597: Get all configured tools
    get_all_tools,
    
    # Date utility - Lines 872-879: Get formatted current date
    get_today_str,
    
    # Supporting utilities for error handling
    get_api_key_for_model,          # Lines 892-914: Get API keys from config or env
    is_token_limit_exceeded,         # Lines 665-701: Detect token limit errors
    get_model_token_limit,           # Lines 831-846: Look up model's token limit
    remove_up_to_last_ai_message,    # Lines 848-866: Truncate messages for retry
    anthropic_websearch_called,      # Lines 607-637: Detect Anthropic native search usage
    openai_websearch_called,         # Lines 639-658: Detect OpenAI native search usage
    get_notes_from_tool_calls,       # Lines 599-601: Extract notes from tool messages
)

### ❓ Question 2:  

What are the advantages and disadvantages of importing these components instead of including them in the notebook?

## Task 3: Configuration System

The configuration system controls:

### Research Behavior
- **allow_clarification** - Whether to ask clarifying questions before research
- **max_concurrent_research_units** - How many parallel researchers can run (default: 5)
- **max_researcher_iterations** - How many times supervisor can delegate research (default: 6)
- **max_react_tool_calls** - Tool call limit per researcher (default: 10)

### Model Configuration
- **research_model** - Model for research and supervision (we'll use Anthropic)
- **compression_model** - Model for synthesizing findings
- **final_report_model** - Model for writing the final report
- **summarization_model** - Model for summarizing web search results

### Search Configuration
- **search_api** - Which search API to use (ANTHROPIC, TAVILY, or NONE)
- **max_content_length** - Character limit before summarization

Defined in [`open_deep_library/configuration.py`](open_deep_library/configuration.py)

In [4]:
# Import configuration from the library
from open_deep_library.configuration import (
    Configuration,    # Lines 38-247: Main configuration class with all settings
    SearchAPI,        # Lines 11-17: Enum for search API options (ANTHROPIC, TAVILY, NONE)
)

## Task 4: Prompt Templates

The system uses carefully engineered prompts for each phase:

### Phase 1: Clarification
**clarify_with_user_instructions** - Analyzes if the research scope is clear or needs clarification

### Phase 2: Research Brief
**transform_messages_into_research_topic_prompt** - Converts user messages into a detailed research brief

### Phase 3: Supervisor
**lead_researcher_prompt** - System prompt for the supervisor that manages delegation strategy

### Phase 4: Researcher
**research_system_prompt** - System prompt for individual researchers conducting focused research

### Phase 5: Compression
**compress_research_system_prompt** - Prompt for synthesizing research findings without losing information

### Phase 6: Final Report
**final_report_generation_prompt** - Comprehensive prompt for writing the final report

All prompts are defined in [`open_deep_library/prompts.py`](open_deep_library/prompts.py)

In [5]:
# Import prompt templates from the library
from open_deep_library.prompts import (
    clarify_with_user_instructions,                    # Lines 3-41: Ask clarifying questions
    transform_messages_into_research_topic_prompt,     # Lines 44-77: Generate research brief
    lead_researcher_prompt,                            # Lines 79-136: Supervisor system prompt
    research_system_prompt,                            # Lines 138-183: Researcher system prompt
    compress_research_system_prompt,                   # Lines 186-222: Research compression prompt
    final_report_generation_prompt,                    # Lines 228-308: Final report generation
)

## Task 5: Node Functions - The Building Blocks

Now let's look at the node functions that make up our graph. We'll import them from the library and understand what each does.

### The Complete Research Workflow

The workflow consists of 8 key nodes organized into 3 subgraphs:

1. **Main Graph Nodes:**
   - `clarify_with_user` - Entry point that checks if clarification is needed
   - `write_research_brief` - Transforms user input into structured research brief
   - `final_report_generation` - Synthesizes all research into final report

2. **Supervisor Subgraph Nodes:**
   - `supervisor` - Lead researcher that plans and delegates
   - `supervisor_tools` - Executes supervisor's tool calls (delegation, reflection)

3. **Researcher Subgraph Nodes:**
   - `researcher` - Individual researcher conducting focused research
   - `researcher_tools` - Executes researcher's tool calls (search, reflection)
   - `compress_research` - Synthesizes researcher's findings

All nodes are defined in [`open_deep_library/deep_researcher.py`](open_deep_library/deep_researcher.py)

### Node 1: clarify_with_user

**Purpose:** Analyzes user messages and asks clarifying questions if the research scope is unclear.

**Key Steps:**
1. Check if clarification is enabled in configuration
2. Use structured output to analyze if clarification is needed
3. If needed, end with a clarifying question for the user
4. If not needed, proceed to research brief with verification message

**Implementation:** [`open_deep_library/deep_researcher.py` lines 60-115](open_deep_library/deep_researcher.py#L60-L115)

In [6]:
# Import the clarify_with_user node
from open_deep_library.deep_researcher import clarify_with_user

### Node 2: write_research_brief

**Purpose:** Transforms user messages into a structured research brief for the supervisor.

**Key Steps:**
1. Use structured output to generate detailed research brief from messages
2. Initialize supervisor with system prompt and research brief
3. Set up supervisor messages with proper context

**Why this matters:** A well-structured research brief helps the supervisor make better delegation decisions.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 118-175](open_deep_library/deep_researcher.py#L118-L175)

In [7]:
# Import the write_research_brief node
from open_deep_library.deep_researcher import write_research_brief

### Node 3: supervisor

**Purpose:** Lead research supervisor that plans research strategy and delegates to sub-researchers.

**Key Steps:**
1. Configure model with three tools:
   - `ConductResearch` - Delegate research to a sub-agent
   - `ResearchComplete` - Signal that research is done
   - `think_tool` - Strategic reflection before decisions
2. Generate response based on current context
3. Increment research iteration count
4. Proceed to tool execution

**Decision Making:** The supervisor uses `think_tool` to reflect before delegating research, ensuring thoughtful decomposition of the research question.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 178-223](open_deep_library/deep_researcher.py#L178-L223)

In [8]:
# Import the supervisor node (from supervisor subgraph)
from open_deep_library.deep_researcher import supervisor

### Node 4: supervisor_tools

**Purpose:** Executes the supervisor's tool calls, including strategic thinking and research delegation.

**Key Steps:**
1. Check exit conditions:
   - Exceeded maximum iterations
   - No tool calls made
   - `ResearchComplete` called
2. Process `think_tool` calls for strategic reflection
3. Execute `ConductResearch` calls in parallel:
   - Spawn researcher subgraphs for each delegation
   - Limit to `max_concurrent_research_units` (default: 5)
   - Gather all results asynchronously
4. Aggregate findings and return to supervisor

**Parallel Execution:** This is where the magic happens - multiple researchers work simultaneously on different aspects of the research question.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 225-349](open_deep_library/deep_researcher.py#L225-L349)

In [9]:
# Import the supervisor_tools node
from open_deep_library.deep_researcher import supervisor_tools

### Node 5: researcher

**Purpose:** Individual researcher that conducts focused research on a specific topic.

**Key Steps:**
1. Load all available tools (search, MCP, reflection)
2. Configure model with tools and researcher system prompt
3. Generate response with tool calls
4. Increment tool call iteration count

**ReAct Pattern:** Researchers use `think_tool` to reflect after each search, deciding whether to continue or provide their answer.

**Available Tools:**
- Search tools (Tavily or Anthropic native search)
- `think_tool` for strategic reflection
- `ResearchComplete` to signal completion
- MCP tools (if configured)

**Implementation:** [`open_deep_library/deep_researcher.py` lines 365-424](open_deep_library/deep_researcher.py#L365-L424)

In [10]:
# Import the researcher node (from researcher subgraph)
from open_deep_library.deep_researcher import researcher

### Node 6: researcher_tools

**Purpose:** Executes the researcher's tool calls, including searches and strategic reflection.

**Key Steps:**
1. Check early exit conditions (no tool calls, native search used)
2. Execute all tool calls in parallel:
   - Search tools fetch and summarize web content
   - `think_tool` records strategic reflections
   - MCP tools execute external integrations
3. Check late exit conditions:
   - Exceeded `max_react_tool_calls` (default: 10)
   - `ResearchComplete` called
4. Continue research loop or proceed to compression

**Error Handling:** Safely handles tool execution errors and continues with available results.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 435-509](open_deep_library/deep_researcher.py#L435-L509)

In [11]:
# Import the researcher_tools node
from open_deep_library.deep_researcher import researcher_tools

### Node 7: compress_research

**Purpose:** Compresses and synthesizes research findings into a concise, structured summary.

**Key Steps:**
1. Configure compression model
2. Add compression instruction to messages
3. Attempt compression with retry logic:
   - If token limit exceeded, remove older messages
   - Retry up to 3 times
4. Extract raw notes from tool and AI messages
5. Return compressed research and raw notes

**Why Compression?** Researchers may accumulate lots of tool outputs and reflections. Compression ensures:
- All important information is preserved
- Redundant information is deduplicated
- Content stays within token limits for the final report

**Token Limit Handling:** Gracefully handles token limit errors by progressively truncating messages.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 511-585](open_deep_library/deep_researcher.py#L511-L585)

In [12]:
# Import the compress_research node
from open_deep_library.deep_researcher import compress_research

### Node 8: final_report_generation

**Purpose:** Generates the final comprehensive research report from all collected findings.

**Key Steps:**
1. Extract all notes from completed research
2. Configure final report model
3. Attempt report generation with retry logic:
   - If token limit exceeded, truncate findings by 10%
   - Retry up to 3 times
4. Return final report or error message

**Token Limit Strategy:**
- First retry: Use model's token limit × 4 as character limit
- Subsequent retries: Reduce by 10% each time
- Graceful degradation with helpful error messages

**Report Quality:** The prompt guides the model to create well-structured reports with:
- Proper headings and sections
- Inline citations
- Comprehensive coverage of all findings
- Sources section at the end

**Implementation:** [`open_deep_library/deep_researcher.py` lines 607-697](open_deep_library/deep_researcher.py#L607-L697)

In [13]:
# Import the final_report_generation node
from open_deep_library.deep_researcher import final_report_generation

## Task 6: Graph Construction - Putting It All Together

The system is organized into three interconnected graphs:

### 1. Researcher Subgraph (Bottom Level)
Handles individual focused research on a specific topic:
```
START → researcher → researcher_tools → compress_research → END
               ↑            ↓
               └────────────┘ (loops until max iterations or ResearchComplete)
```

### 2. Supervisor Subgraph (Middle Level)
Manages research delegation and coordination:
```
START → supervisor → supervisor_tools → END
            ↑              ↓
            └──────────────┘ (loops until max iterations or ResearchComplete)
            
supervisor_tools spawns multiple researcher_subgraphs in parallel
```

### 3. Main Deep Researcher Graph (Top Level)
Orchestrates the complete research workflow:
```
START → clarify_with_user → write_research_brief → research_supervisor → final_report_generation → END
                 ↓                                       (supervisor_subgraph)
               (may end early if clarification needed)
```

Let's import the compiled graphs from the library.

In [14]:
# Import the pre-compiled graphs from the library
from open_deep_library.deep_researcher import (
    # Bottom level: Individual researcher workflow
    researcher_subgraph,    # Lines 588-605: researcher → researcher_tools → compress_research
    
    # Middle level: Supervisor coordination
    supervisor_subgraph,    # Lines 351-363: supervisor → supervisor_tools (spawns researchers)
    
    # Top level: Complete research workflow
    deep_researcher,        # Lines 699-719: Main graph with all phases
)

## Why This Architecture?

### Advantages of Supervisor-Researcher Delegation

1. **Dynamic Task Decomposition**
   - Unlike section-based approaches with predefined structure, the supervisor can break down research based on the actual question
   - Adapts to different types of research (comparisons, lists, deep dives, etc.)

2. **Parallel Execution**
   - Multiple researchers work simultaneously on different aspects
   - Much faster than sequential section processing
   - Configurable parallelism (1-20 concurrent researchers)

3. **ReAct Pattern for Quality**
   - Researchers use `think_tool` to reflect after each search
   - Prevents excessive searching and improves search quality
   - Natural stopping conditions based on information sufficiency

4. **Flexible Tool Integration**
   - Easy to add MCP tools for specialized research
   - Supports multiple search APIs (Anthropic, Tavily)
   - Each researcher can use different tool combinations

5. **Graceful Token Limit Handling**
   - Compression prevents token overflow
   - Progressive truncation in final report generation
   - Research can scale to arbitrary depths

### Trade-offs

- **Complexity:** More moving parts than section-based approach
- **Cost:** Parallel researchers use more tokens (but faster)
- **Unpredictability:** Research structure emerges dynamically

## Task 7: Running the Deep Researcher

Now let's see the system in action! We'll use it to analyze a PDF document about how people use AI.

### Setup

We need to:
1. Load the PDF document
2. Configure the execution with Anthropic settings
3. Run the research workflow

In [15]:
# Load the PDF document
from pathlib import Path
import PyPDF2

def load_pdf(pdf_path: str) -> str:
    """Load and extract text from PDF."""
    pdf_text = ""
    with open(pdf_path, 'rb') as file:
        pdf_reader = PyPDF2.PdfReader(file)
        for page in pdf_reader.pages:
            pdf_text += page.extract_text() + "\n\n"
    return pdf_text

# Load the PDF about how people use AI
pdf_path = "data/howpeopleuseai.pdf"
pdf_content = load_pdf(pdf_path)

print(f"Loaded PDF with {len(pdf_content)} characters")
print(f"First 500 characters:\n{pdf_content[:500]}...")

Loaded PDF with 112460 characters
First 500 characters:
NBER WORKING PAPER SERIES
HOW PEOPLE USE CHATGPT
Aaron Chatterji
Thomas Cunningham
David J. Deming
Zoe Hitzig
Christopher Ong
Carl Yan Shan
Kevin Wadman
Working Paper 34255
http://www.nber.org/papers/w34255
NATIONAL BUREAU OF ECONOMIC RESEARCH
1050 Massachusetts Avenue
Cambridge, MA 02138
September 2025
We acknowledge help and comments from Joshua Achiam, Hemanth Asirvatham, Ryan 
Beiermeister,  Rachel Brown, Cassandra Duchan Solis, Jason Kwon, Elliott Mokski, Kevin Rao, 
Harrison Satcher,  Gawe...


In [16]:
# Set up the graph with Anthropic configuration
from IPython.display import Markdown, display
import uuid

# Note: deep_researcher is already compiled from the library
# For this demo, we'll use it directly without additional checkpointing
graph = deep_researcher

print("✓ Graph ready for execution")
print("  (Note: The graph is pre-compiled from the library)")

✓ Graph ready for execution
  (Note: The graph is pre-compiled from the library)


### Configuration for Anthropic

We'll configure the system to use:
- **Claude Sonnet 4** for all research, supervision, and report generation
- **Tavily** for web search (you can also use Anthropic's native search)
- **Moderate parallelism** (3 concurrent researchers)
- **Clarification enabled** (will ask if research scope is unclear)

In [17]:
# Configure for Anthropic with moderate settings
config = {
    "configurable": {
        # Model configuration - using Claude Sonnet 4 for everything
        "research_model": "anthropic:claude-sonnet-4-20250514",
        "research_model_max_tokens": 10000,
        
        "compression_model": "anthropic:claude-sonnet-4-20250514",
        "compression_model_max_tokens": 8192,
        
        "final_report_model": "anthropic:claude-sonnet-4-20250514",
        "final_report_model_max_tokens": 10000,
        
        "summarization_model": "anthropic:claude-sonnet-4-20250514",
        "summarization_model_max_tokens": 8192,
        
        # Research behavior
        "allow_clarification": True,
        "max_concurrent_research_units": 1,  # 1 parallel researchers
        "max_researcher_iterations": 2,      # Supervisor can delegate up to 2 times
        "max_react_tool_calls": 3,           # Each researcher can make up to 3 tool calls
        
        # Search configuration
        "search_api": "tavily",  # Using Tavily for web search
        "max_content_length": 50000,
        
        # Thread ID for this conversation
        "thread_id": str(uuid.uuid4())
    }
}

print("✓ Configuration ready")
print(f"  - Research Model: Claude Sonnet 4")
print(f"  - Max Concurrent Researchers: 3")
print(f"  - Max Iterations: 4")
print(f"  - Search API: Tavily")

✓ Configuration ready
  - Research Model: Claude Sonnet 4
  - Max Concurrent Researchers: 3
  - Max Iterations: 4
  - Search API: Tavily


### Execute the Research

Now let's run the research! We'll ask the system to analyze the PDF and provide insights about how people use AI.

The workflow will:
1. **Clarify** - Check if the request is clear (may skip if obvious)
2. **Research Brief** - Transform our request into a structured brief
3. **Supervisor** - Plan research strategy and delegate to researchers
4. **Parallel Research** - Multiple researchers gather information simultaneously
5. **Compression** - Each researcher synthesizes their findings
6. **Final Report** - All findings combined into comprehensive report

In [18]:
# Create our research request with PDF context
research_request = f"""
I have a PDF document about how people use AI. Please analyze this document and provide insights about:

1. What are the main findings about how people are using AI?
2. What are the most common use cases?
3. What trends or patterns emerge from the data?

Here's the PDF content:

{pdf_content[:10000]}  # First 10k chars to stay within limits

...[content truncated for context window]
"""

# Execute the graph
async def run_research():
    """Run the research workflow and display results."""
    print("Starting research workflow...\n")
    
    async for event in graph.astream(
        {"messages": [{"role": "user", "content": research_request}]},
        config,
        stream_mode="updates"
    ):
        # Display each step
        for node_name, node_output in event.items():
            print(f"\n{'='*60}")
            print(f"Node: {node_name}")
            print(f"{'='*60}")
            
            if node_name == "clarify_with_user":
                if "messages" in node_output:
                    last_msg = node_output["messages"][-1]
                    print(f"\n{last_msg.content}")
            
            elif node_name == "write_research_brief":
                if "research_brief" in node_output:
                    print(f"\nResearch Brief Generated:")
                    print(f"{node_output['research_brief'][:500]}...")
            
            elif node_name == "supervisor":
                print(f"\nSupervisor planning research strategy...")
                if "supervisor_messages" in node_output:
                    last_msg = node_output["supervisor_messages"][-1]
                    if hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:
                        print(f"Tool calls: {len(last_msg.tool_calls)}")
                        for tc in last_msg.tool_calls:
                            print(f"  - {tc['name']}")
            
            elif node_name == "supervisor_tools":
                print(f"\nExecuting supervisor's tool calls...")
                if "notes" in node_output:
                    print(f"Research notes collected: {len(node_output['notes'])}")
            
            elif node_name == "final_report_generation":
                if "final_report" in node_output:
                    print(f"\n" + "="*60)
                    print("FINAL REPORT GENERATED")
                    print("="*60 + "\n")
                    display(Markdown(node_output["final_report"]))
    
    print("\n" + "="*60)
    print("Research workflow completed!")
    print("="*60)

# Run the research
await run_research()

Starting research workflow...


Node: clarify_with_user

I have sufficient information to proceed with analyzing the NBER working paper "How People Use ChatGPT." I understand you want insights on: (1) main findings about how people are using AI, (2) most common use cases, and (3) trends and patterns from the data. The document provides comprehensive data on ChatGPT usage from November 2022 through July 2025, including usage patterns, demographics, work vs. non-work applications, and conversation topics. I will now analyze this research and provide you with the requested insights.

Node: write_research_brief

Research Brief Generated:
I need a comprehensive analysis of the NBER working paper "How People Use ChatGPT" (Working Paper No. 34255, September 2025) by Aaron Chatterji, Thomas Cunningham, David J. Deming, Zoe Hitzig, Christopher Ong, Carl Yan Shan, and Kevin Wadman. Specifically, I want insights into three key areas: (1) What are the main findings about how people are using AI ba


Node: research_supervisor

Node: final_report_generation

FINAL REPORT GENERATED



# Comprehensive Analysis of NBER Working Paper "How People Use ChatGPT"

## Executive Summary

The NBER working paper "How People Use ChatGPT" (Working Paper No. 34255) represents the most comprehensive analysis of AI chatbot usage patterns to date, examining ChatGPT adoption and usage from its November 2022 launch through July 2025. The study reveals that ChatGPT achieved unprecedented global adoption, reaching approximately 10% of the world's adult population with over 750 million weekly active users by September 2025. The research employs sophisticated privacy-preserving methodologies to analyze over 1.1 million de-identified conversations, revealing significant shifts in usage patterns, demographic adoption trends, and the evolution of AI's role in both professional and personal contexts.

## Main Findings About AI Usage Patterns

### Unprecedented Adoption Rates and Growth Trajectory

ChatGPT's adoption represents the fastest technology diffusion in recorded history. Within one year of its November 2022 launch, ChatGPT reached 100 million weekly active users by early November 2023. By July 2025, the platform had achieved a staggering scale: over 750 million weekly active users sending more than 2.6 billion messages daily, equivalent to over 30,000 messages per second. The number of weekly active users has been doubling every 7-8 months since launch, with total message volume increasing by 5.8x in the final year of the study period alone [1][2].

The scale becomes even more remarkable when compared to established platforms. ChatGPT reached 1 billion daily messages in December 2024, less than two years after release, while Google Search required eight years to reach 1 billion daily searches after its 1999 public launch [2][7].

### Demographic Evolution and Narrowing Gaps

The demographic composition of ChatGPT users has undergone dramatic transformation since launch. Early adopters were overwhelmingly male, with approximately 80% having typically masculine first names in the initial months. However, this gender gap has essentially disappeared by July 2025, with 52% of active users now having typically feminine first names, suggesting complete gender parity or even a slight female majority [1][2][4][5][7][8].

Age patterns reveal ChatGPT's particular appeal to younger demographics, with nearly half of all adult messages sent by users under 26. Among adults, the 18-25 age cohort represents almost half of all message volume, making ChatGPT especially popular among the youngest adult segment [1][4][9].

### Global Adoption and Economic Patterns

A significant finding relates to geographic and economic adoption patterns. ChatGPT usage has grown relatively faster in low- and middle-income countries, with growth rates in the lowest income countries exceeding those in highest income countries by over 4x as of May 2025. Countries at the 50th percentile of GDP per capita now demonstrate usage rates similar to those at the 90th percentile, indicating rapid global democratization of AI access [1][4][10].

### User Engagement and Retention Patterns

User engagement has consistently increased across all cohorts, with substantial growth beginning in early 2025. Early adopters from Q1 2023 are now sending 40% more messages per day than they did two years earlier. This pattern holds across all user cohorts regardless of signup date, suggesting both technological improvements and users' expanding discovery of new applications [2][7].

## Most Common Use Cases: The Three Dominant Categories

### Practical Guidance (29% of Overall Usage)

Practical Guidance emerged as the most prevalent use case, maintaining a consistent 29% share throughout the study period. This category encompasses highly customized, interactive assistance that adapts based on conversation flow and follow-up questions. Key subcategories include:

- **Tutoring and Teaching**: Representing 10.2% of all user messages and 36% of Practical Guidance messages
- **How-to Advice**: Accounting for 8.5% of total usage and 30% of Practical Guidance
- **Creative Ideation**: Supporting brainstorming and creative problem-solving processes

The distinguishing characteristic of Practical Guidance is its personalized, adaptive nature, contrasting with static information retrieval [5].

### Seeking Information (24% of Usage by July 2025)

Seeking Information functions as a direct substitute for traditional web search, encompassing searches for people, current events, products, and recipes. This category demonstrated significant growth, expanding from 14% to 24% of all usage between July 2024 and July 2025. The category provides factual information that should be consistent across users, distinguishing it from the personalized nature of Practical Guidance [5].

### Writing (24% of Usage by July 2025)

Writing represents both novel text generation and modification of existing content, though it has declined from 36% of usage in July 2024 to 24% a year later. The category encompasses five primary subcategories in order of frequency:

1. **Editing or Critiquing Provided Text**
2. **Personal Writing or Communication**
3. **Translation**
4. **Argument or Summary Generation**
5. **Writing Fiction**

Notably, three of these five categories involve modifying user-provided text rather than creating entirely new content, constituting two-thirds of all Writing conversations [5].

### Work Context Applications

In professional settings, Writing dominates usage patterns, accounting for 40% of work-related messages as of June 2025. This reflects ChatGPT's unique capability to generate digital outputs compared to traditional search engines. Work usage correlates strongly with education levels and professional occupations, with educated users in highly-paid professional roles substantially more likely to use ChatGPT for work purposes [1][4].

Technical Help in work contexts has declined significantly from 18% of work-related messages in July 2024 to just over 10% in July 2025. Surprisingly, computer programming represents only 4.2% of ChatGPT messages, contrasting sharply with 33% of work-related Claude conversations reported in other studies [5].

### Non-Work Context Applications

Non-work usage has experienced explosive growth, expanding from 53% to over 70% of all usage between June 2024 and June 2025. This shift primarily reflects changing usage patterns within existing user cohorts rather than compositional changes in new users. The research suggests that AI's impact on home production may equal or exceed its workplace productivity effects [1][5].

## Trends and Patterns from the Data

### The Dramatic Work-to-Non-Work Usage Shift

The most striking trend revealed in the study is the fundamental shift in usage context. In June 2024, work-related messages comprised 213 million daily messages (47% of total), while non-work messages totaled 238 million (53%). By June 2025, these proportions had dramatically reversed: work messages reached 716 million (27% of total) while non-work messages exploded to 1,911 million (73% of total) [1][4][9].

This transformation suggests that while initial economic analysis focused on AI's workplace productivity impact, its influence on personal and home production activities may be equally or more significant [1].

### User Intent Classification Evolution

The research introduced an innovative taxonomy classifying messages by user intent: Asking (seeking information), Doing (performing tasks), and Expressing (creative or emotional expression). In July 2024, usage was roughly split between Asking and Doing (approximately 46% each), with 8% Expressing. By June 2025, the distribution shifted to 51.6% Asking, 34.6% Doing, and 13.8% Expressing, indicating users increasingly seek information and advice rather than task execution [5][9].

However, work contexts show different patterns, with 56% of work-related messages classified as Doing, and nearly three-quarters of those involving Writing tasks [1].

### Conversation Topics Evolution

Several conversation topics experienced significant changes over the study period:

- **Technical Help** declined from 12% of all usage in July 2024 to approximately 5% a year later
- **Multimedia** grew from 2% to over 7%, with a notable spike in April 2025 following ChatGPT's new image-generation capabilities
- The three dominant categories (Practical Guidance, Seeking Information, Writing) have remained consistently prominent, collectively accounting for nearly 80% of all conversations [5]

### Geographic and Economic Adoption Patterns

The data reveals accelerating adoption in developing economies, with growth rates in lower-income countries substantially exceeding those in wealthier nations. This pattern suggests AI technology is rapidly democratizing globally, potentially reducing digital divides rather than exacerbating them [2][7].

## Methodology and Research Innovation

### Privacy-Preserving Automated Classification Pipeline

The study's methodological innovation lies in its privacy-preserving approach to analyzing sensitive user data. The researchers developed an entirely automated classification system where no human researcher ever accessed raw user messages. The process involves:

1. **Privacy Filter**: An internal LLM-based tool strips personally identifiable information (PII) from messages
2. **Automated Classification**: De-identified messages are classified using LLM-based automated classifiers
3. **Technical Safeguards**: Interfaces prevent accidental access to underlying message text

This methodology enables comprehensive analysis while maintaining strict privacy protection [1][5].

### Secure Data Clean Room Protocol

For employment and education analysis, the researchers employed a secure Data Clean Room (DCR) that permitted only pre-approved aggregate computations across independently held datasets. The system enforced strict aggregation limits, approving only code returning cells with at least 100 users, ensuring no individual data was visible to researchers [5].

### Validation and Transparency

The research team validated classification prompts by comparing model decisions against human-judged classifications using the publicly available WildChat dataset. They classified 100,000 public WildChat messages and included this data in their replication package for transparency. The study analyzed approximately 1.1 million de-identified messages from consumer ChatGPT users, with some analyses extending to 1.58 million messages [5].

## Economic and Social Implications

### Consumer Surplus and Economic Value

The findings align with economic research by Collis and Brynjolfsson (2025), who estimate consumer surplus from generative AI at least $97 billion in 2024 in the US alone. The study concludes that ChatGPT provides economic value primarily through decision support, particularly important in knowledge-intensive jobs [1][6].

### Comparison with Other AI Platforms

The research provides important comparative context with other AI platforms. The relatively low proportion of coding-related usage (4.2% of ChatGPT messages) contrasts sharply with 33% of work-related Claude conversations, suggesting different platforms serve distinct user needs and demographics [1][5].

### Social and Emotional Usage Patterns

Contrary to some expectations, the study found relatively limited usage for companionship or social-emotional purposes. Only 1.9% of messages relate to Relationships and Personal Reflection, and 0.4% to Games and Role Play. This contrasts with other research suggesting Therapy/Companionship as the most prevalent generative AI use case [1][5].

## Future Research Directions and Limitations

The study represents foundational research in understanding AI adoption and usage patterns, but several areas warrant further investigation. The focus on ChatGPT, while representing the largest user base, may not capture usage patterns across all AI platforms. Additionally, the rapid evolution of AI capabilities means usage patterns may continue shifting significantly.

The research methodology, while innovative in privacy protection, relies on automated classification systems that, despite validation, may miss nuanced usage patterns that human analysis might detect. Future research might explore cross-platform usage patterns, longitudinal studies of individual user behavior, and deeper analysis of demographic and geographic usage variations.

The study's findings have significant implications for understanding AI's role in the economy and society, suggesting that personal and home production impacts may equal or exceed workplace productivity effects, fundamentally reshaping how economists and policymakers should approach AI impact assessment.

### Sources

[1] How People Use ChatGPT - SSRN: https://papers.ssrn.com/sol3/papers.cfm?abstract_id=5487080
[2] How People Use ChatGPT - by David Deming - Forked Lightning: https://forklightning.substack.com/p/how-people-use-chatgpt
[3] David J. Deming | NBER: https://www.nber.org/people/david_deming
[4] How People Use ChatGPT | NBER: https://www.nber.org/papers/w34255
[5] [PDF] How People Use ChatGPT - OpenAI: https://cdn.openai.com/pdf/a253471f-8260-40c6-a2cc-aa93fe9f142e/economic-research-chatgpt-usage-paper.pdf
[6] How People Are Really Using ChatGPT - Mike Jeffs: https://mikejeffs.com/blog/how-people-are-really-using-chatgpt/
[7] how people use chatgpt - Medium: https://medium.com/@danny_54172/how-people-use-chatgpt-842c0427182a
[8] How People Actually Use ChatGPT — What 1.5M Conversations Tell Us: https://medium.com/@adnanmasood/how-people-actually-use-chatgpt-what-1-5m-conversations-tell-us-about-the-next-decade-of-software-ea603212b458
[9] OpenAI's ChatGPT study reveals growth, usage patterns - LinkedIn: https://www.linkedin.com/posts/saudhashimi_openai-chatgpt-usage-research-activity-7373407019422105601-fvHF
[10] How people are using ChatGPT | OpenAI: https://openai.com/index/how-people-are-using-chatgpt/


Research workflow completed!


## Understanding the Output

Let's break down what happened:

### Phase 1: Clarification
The system checked if your request was clear. Since you provided a PDF and specific questions, it likely proceeded without clarification.

### Phase 2: Research Brief
Your request was transformed into a detailed research brief that guides the supervisor's delegation strategy.

### Phase 3: Supervisor Delegation
The supervisor analyzed the brief and decided how to break down the research:
- Used `think_tool` to plan strategy
- Called `ConductResearch` multiple times to delegate to parallel researchers
- Each delegation specified a focused research topic

### Phase 4: Parallel Research
Multiple researchers worked simultaneously:
- Each researcher used web search tools to gather information
- Used `think_tool` to reflect after each search
- Decided when they had enough information
- Compressed their findings into clean summaries

### Phase 5: Final Report
All research findings were synthesized into a comprehensive report with:
- Well-structured sections
- Inline citations
- Sources listed at the end
- Balanced coverage of all findings

## Activity #1 Solution: Quick Configuration Experiments




In [21]:
# Simple helper function to run experiments quickly
import uuid
from IPython.display import Markdown, display

async def quick_experiment(config_name, config_overrides, research_question):
    """Run a quick research experiment with given configuration."""
    
    # Base configuration
    config = {
        "configurable": {
            "research_model": "anthropic:claude-sonnet-4-20250514",
            "research_model_max_tokens": 10000,
            "compression_model": "anthropic:claude-sonnet-4-20250514",
            "compression_model_max_tokens": 8192,
            "final_report_model": "anthropic:claude-sonnet-4-20250514",
            "final_report_model_max_tokens": 10000,
            "summarization_model": "anthropic:claude-sonnet-4-20250514",
            "summarization_model_max_tokens": 8192,
            "allow_clarification": False,  # Skip clarification for speed
            "max_content_length": 50000,
            "thread_id": str(uuid.uuid4()),
            # Defaults that can be overridden
            "max_concurrent_research_units": 1,
            "max_researcher_iterations": 2,
            "max_react_tool_calls": 3,
            "search_api": "tavily"
        }
    }
    
    # Apply overrides
    config["configurable"].update(config_overrides)
    
    print(f"\n{'='*70}")
    print(f"EXPERIMENT: {config_name}")
    print(f"{'='*70}")
    print(f"Settings: {config_overrides}")
    print(f"{'='*70}\n")
    
    # Run research
    async for event in graph.astream(
        {"messages": [{"role": "user", "content": research_question}]},
        config,
        stream_mode="updates"
    ):
        for node_name, node_output in event.items():
            if node_name == "final_report_generation":
                if "final_report" in node_output:
                    print("\n📊 FINAL REPORT:\n")
                    display(Markdown(node_output["final_report"]))
                    print(f"\n{'='*70}\n")

print("✓ Quick experiment helper ready")


✓ Quick experiment helper ready


### Experiment 1: Baseline (Minimal Settings)


In [22]:
# Experiment 1: Baseline - minimal settings for quick results
await quick_experiment(
    "Baseline (Minimal)",
    {
        "max_concurrent_research_units": 1,
        "max_researcher_iterations": 2,
        "max_react_tool_calls": 3,
        "search_api": "tavily"
    },
    "What are the top 3 benefits of using LangGraph for AI agents?"
)



EXPERIMENT: Baseline (Minimal)
Settings: {'max_concurrent_research_units': 1, 'max_researcher_iterations': 2, 'max_react_tool_calls': 3, 'search_api': 'tavily'}




📊 FINAL REPORT:



Error generating final report: Error code: 429 - {'type': 'error', 'error': {'type': 'rate_limit_error', 'message': 'This request would exceed the rate limit for your organization (e5ddd8a0-7acb-4b21-8f9f-7d1c6eca4d4b) of 30,000 input tokens per minute. For details, refer to: https://docs.claude.com/en/api/rate-limits. You can see the response headers for current usage. Please reduce the prompt length or the maximum tokens requested, or try again later. You may also contact sales at https://www.anthropic.com/contact-sales to discuss your options for a rate limit increase.'}, 'request_id': 'req_011CU89kPAMep3zYjbddKMnG'}

### Experiment 2: Increased Parallelism


In [23]:
# Experiment 2: Increased Parallelism - more researchers working simultaneously
await quick_experiment(
    "Increased Parallelism",
    {
        "max_concurrent_research_units": 5,  # More parallel researchers
        "max_researcher_iterations": 2,
        "max_react_tool_calls": 3,
        "search_api": "tavily"
    },
    "What are the top 3 benefits of using LangGraph for AI agents?"
)



EXPERIMENT: Increased Parallelism
Settings: {'max_concurrent_research_units': 5, 'max_researcher_iterations': 2, 'max_react_tool_calls': 3, 'search_api': 'tavily'}




📊 FINAL REPORT:



Error generating final report: Error code: 429 - {'type': 'error', 'error': {'type': 'rate_limit_error', 'message': 'This request would exceed the rate limit for your organization (e5ddd8a0-7acb-4b21-8f9f-7d1c6eca4d4b) of 30,000 input tokens per minute. For details, refer to: https://docs.claude.com/en/api/rate-limits. You can see the response headers for current usage. Please reduce the prompt length or the maximum tokens requested, or try again later. You may also contact sales at https://www.anthropic.com/contact-sales to discuss your options for a rate limit increase.'}, 'request_id': 'req_011CU89zK5vkPVeJCLqh5rkJ'}

### Experiment 3: Deeper Research (More Iterations)


In [24]:
# Experiment 3: Deeper Research - more iterations and tool calls
await quick_experiment(
    "Deeper Research",
    {
        "max_concurrent_research_units": 1,
        "max_researcher_iterations": 5,  # More supervisor iterations
        "max_react_tool_calls": 8,       # More tool calls per researcher
        "search_api": "tavily"
    },
    "What are the top 3 benefits of using LangGraph for AI agents?"
)



EXPERIMENT: Deeper Research
Settings: {'max_concurrent_research_units': 1, 'max_researcher_iterations': 5, 'max_react_tool_calls': 8, 'search_api': 'tavily'}




📊 FINAL REPORT:



Error generating final report: Error code: 429 - {'type': 'error', 'error': {'type': 'rate_limit_error', 'message': 'This request would exceed the rate limit for your organization (e5ddd8a0-7acb-4b21-8f9f-7d1c6eca4d4b) of 30,000 input tokens per minute. For details, refer to: https://docs.claude.com/en/api/rate-limits. You can see the response headers for current usage. Please reduce the prompt length or the maximum tokens requested, or try again later. You may also contact sales at https://www.anthropic.com/contact-sales to discuss your options for a rate limit increase.'}, 'request_id': 'req_011CU8AMekhSTKJvFRjECLLd'}

### Experiment 4: Anthropic Native Search


In [25]:
# Experiment 4: Anthropic Native Search - use Claude's built-in search
await quick_experiment(
    "Anthropic Native Search",
    {
        "max_concurrent_research_units": 1,
        "max_researcher_iterations": 2,
        "max_react_tool_calls": 3,
        "search_api": "anthropic"  # Use Claude's native search instead of Tavily
    },
    "What are the top 3 benefits of using LangGraph for AI agents?"
)



EXPERIMENT: Anthropic Native Search
Settings: {'max_concurrent_research_units': 1, 'max_researcher_iterations': 2, 'max_react_tool_calls': 3, 'search_api': 'anthropic'}


📊 FINAL REPORT:



# Top 3 Benefits of Using LangGraph for AI Agent Development

LangGraph has emerged as a powerful framework specifically designed for building sophisticated AI agents, offering unique advantages that set it apart from traditional approaches. As part of the LangChain ecosystem, LangGraph addresses critical challenges in agent development through its graph-based architecture and advanced state management capabilities.

## 1. Stateful Multi-Actor Conversations and Complex Workflow Management

LangGraph's most significant advantage lies in its ability to handle stateful, multi-actor conversations through its graph-based architecture. Unlike linear chain-based approaches, LangGraph allows developers to create complex workflows where AI agents can maintain persistent state across multiple interactions and coordinate with other agents or human users.

The framework implements this through its StateGraph class, which enables developers to define nodes (individual functions or agents) and edges (transitions between states) that can loop, branch, and converge based on dynamic conditions. This is particularly powerful for scenarios where agents need to:

- Maintain conversation history and context across multiple turns
- Coordinate with multiple specialized sub-agents
- Handle complex decision trees with conditional branching
- Resume conversations after interruptions or delays

For example, a customer service AI agent built with LangGraph can maintain the full context of a customer's issue across multiple interactions, escalate to human agents when needed while preserving state, and seamlessly transition back to automated handling. This level of state management is difficult to achieve with traditional linear agent frameworks.

The graph-based approach also enables sophisticated error handling and recovery mechanisms. If one node in the graph fails or produces unsatisfactory results, the agent can backtrack to previous states or take alternative paths without losing the accumulated context and progress.

## 2. Human-in-the-Loop Integration with Seamless Approval Workflows

LangGraph excels in scenarios requiring human oversight and approval through its built-in interrupt and approval mechanisms. This capability is crucial for production AI agents that need human validation before taking certain actions, especially in high-stakes environments like financial services, healthcare, or legal applications.

The framework provides native support for:

- **Interrupt Points**: Developers can define specific nodes where execution pauses automatically, allowing humans to review the agent's planned actions before proceeding
- **Dynamic Approvals**: Human reviewers can not only approve or reject actions but also modify the agent's state or provide additional context
- **Resume Capabilities**: After human intervention, agents can seamlessly resume execution from the exact point of interruption with full context preservation

This human-in-the-loop functionality is implemented at the framework level, meaning developers don't need to build custom approval systems or manage complex state persistence manually. The agent automatically handles the coordination between automated processing and human oversight.

A practical example would be a financial AI agent that can analyze market data and prepare trade recommendations automatically, but requires human approval before executing trades above certain thresholds. LangGraph enables this workflow natively, maintaining all analysis context during the approval process and executing the approved actions with the original reasoning intact.

## 3. Advanced Persistence and Checkpoint System for Production Reliability

LangGraph provides enterprise-grade persistence and checkpointing capabilities that make it particularly suitable for production AI agent deployments. The framework includes a sophisticated checkpoint system that automatically saves the agent's state at defined intervals, enabling robust error recovery and long-running process management.

Key persistence features include:

- **Automatic State Checkpointing**: The system can save complete agent state at any point in the execution graph, including conversation history, intermediate results, and decision context
- **Crash Recovery**: If an agent process terminates unexpectedly, it can resume from the last checkpoint without losing progress or context
- **Long-Running Process Support**: Agents can handle tasks that span hours, days, or weeks, with state persisted across system restarts and updates
- **Audit Trails**: Complete execution history is maintained, providing transparency into agent decision-making processes

The checkpoint system integrates with various storage backends, from simple file systems for development to enterprise databases for production deployments. This flexibility allows teams to choose appropriate persistence mechanisms based on their infrastructure and compliance requirements.

For instance, a research AI agent conducting multi-day literature reviews can save progress after each paper analysis, allowing it to survive system maintenance windows and resume exactly where it left off. Similarly, customer support agents can maintain conversation state across shifts, enabling seamless handoffs between different agent instances or human operators.

This persistence capability, combined with LangGraph's state management, enables the development of truly robust AI agents that can operate reliably in production environments where uptime and consistency are critical requirements.

### Sources

[1] LangGraph Documentation - Introduction and Core Concepts: https://langchain-ai.github.io/langgraph/
[2] LangChain Blog - Introducing LangGraph: https://blog.langchain.dev/langgraph/
[3] LangGraph GitHub Repository: https://github.com/langchain-ai/langgraph
[4] LangGraph Tutorials - Human-in-the-Loop Patterns: https://langchain-ai.github.io/langgraph/tutorials/
[5] LangChain Documentation - Agent Architecture with LangGraph: https://python.langchain.com/docs/concepts/agents

### 📝 Your Observations

**Compare the results above and answer these questions:**

1. **Baseline vs Increased Parallelism**: How did the report differ? Was it faster or more comprehensive?

2. **Baseline vs Deeper Research**: Did more iterations and tool calls result in better quality? More sources?

3. **Tavily vs Anthropic Native Search**: Which search API provided better results? Were there differences in speed or quality?

4. **Overall**: Which configuration worked best for this type of question? Why?

**Write your answers here:**

```
[Your observations go here]
```


### 📊 Summary Comparison Table

| Configuration | Speed | Report Length | # of Sources | Quality | Cost Estimate |
|---------------|-------|---------------|--------------|---------|---------------|
| Baseline      |       |               |              |         |               |
| Parallelism   |       |               |              |         |               |
| Deeper        |       |               |              |         |               |
| Anthropic     |       |               |              |         |               |

**Notes:**
- Speed: Fast / Medium / Slow
- Report Length: Short / Medium / Long
- Quality: Low / Medium / High
- Cost Estimate: Low / Medium / High (based on # of API calls)


### 🚀 Bonus: Custom Experiment (Optional)

Try creating your own configuration combining multiple settings!


In [ ]:
# Bonus: Your custom experiment - combine settings as you like!
# Example: High parallelism + deeper research + Anthropic search

await quick_experiment(
    "Custom: Kitchen Sink",
    {
        "max_concurrent_research_units": 5,  # High parallelism
        "max_researcher_iterations": 5,      # More iterations
        "max_react_tool_calls": 8,           # More tool calls
        "search_api": "anthropic"            # Native search
    },
    "What are the top 3 benefits of using LangGraph for AI agents?"
)


---

## ✅ How to Complete the Assignment

1. **Run cells 43-51** in order (all 4 experiments)
2. **Wait for each to complete** and review the reports
3. **Fill in the observations** in cell 52 comparing the results
4. **Complete the comparison table** in cell 53
5. **Optional**: Run cell 55 for the bonus experiment

**Tips for Quick Results:**
- The simple question about LangGraph will run much faster than the PDF analysis
- Each experiment should take 30-120 seconds
- Watch the console output to see which nodes are running
- Compare the final reports side-by-side

**What to Look For:**
- Did parallel researchers produce more diverse information?
- Did deeper research find more sources or better insights?
- Did Anthropic vs Tavily search produce different quality results?
- Which configuration gives you the best quality/speed trade-off?


#### 🏗️ Activity #1: Try Different Configurations

You can experiment with different settings to see how they affect the research.  You may select three or more of the following settings (or invent your own experiments) and describe the results.

### Increase Parallelism
```python
"max_concurrent_research_units": 10  # More researchers working simultaneously
```

### Deeper Research
```python
"max_researcher_iterations": 8   # Supervisor can delegate more times
"max_react_tool_calls": 15      # Each researcher can search more
```

### Use Anthropic Native Search
```python
"search_api": "anthropic"  # Use Claude's built-in web search
```

### Disable Clarification
```python
"allow_clarification": False  # Skip clarification phase
```

## Key Takeaways

### Architecture Benefits
1. **Dynamic Decomposition** - Research structure emerges from the question, not predefined
2. **Parallel Efficiency** - Multiple researchers work simultaneously
3. **ReAct Quality** - Strategic reflection improves search decisions
4. **Scalability** - Handles token limits gracefully through compression
5. **Flexibility** - Easy to add new tools and capabilities

### When to Use This Pattern
- **Complex research questions** that need multi-angle investigation
- **Comparison tasks** where parallel research on different topics is beneficial
- **Open-ended exploration** where structure should emerge dynamically
- **Time-sensitive research** where parallel execution speeds up results

### When to Use Section-Based Instead
- **Highly structured reports** with predefined format requirements
- **Template-based content** where sections are always the same
- **Sequential dependencies** where later sections depend on earlier ones
- **Budget constraints** where token efficiency is critical

## Next Steps

### Extend the System
1. **Add MCP Tools** - Integrate specialized tools for your domain
2. **Custom Prompts** - Modify prompts for specific research types
3. **Different Models** - Try different Claude versions or mix models
4. **Persistence** - Use a real database for checkpointing instead of memory

### Learn More
- [LangGraph Documentation](https://langchain-ai.github.io/langgraph/)
- [Open Deep Research Repo](https://github.com/langchain-ai/open_deep_research)
- [Anthropic Claude Documentation](https://docs.anthropic.com/)
- [Tavily Search API](https://tavily.com/)

### Deploy
- Use LangGraph Cloud for production deployment
- Add proper error handling and logging
- Implement rate limiting and cost controls
- Monitor research quality and costs